## Notebook 10 — Media Behavior Baseline and Cluster Deviations

This notebook analyzes media behavior patterns across the synthetic U.S. population generated in the previous stages of the Market Kinetics pipeline.

The objective is to identify **media behaviors that characterize each structural cluster** by comparing cluster-level averages against the **national media baseline**.

The analysis proceeds in three steps:

1. Load the individual-level population dataset enriched with projected media behaviors.
2. Compute the **national media baseline** using ACS population weights.
3. Calculate **cluster-level deviations from the national baseline** in order to identify media behaviors that meaningfully distinguish each cluster.

These deviations will later be integrated with the psychological cluster signatures derived in Notebook 08 to construct final audience archetypes.

In [1]:
#import
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# Project root (assumes notebook is inside /notebooks)
PROJECT_ROOT   = Path().resolve().parent
DATA_MODELS    = PROJECT_ROOT / "data" / "societal_models"
DATA_PROCESSED = PROJECT_ROOT / "data" / "societal_processed"

In [3]:
df = pd.read_parquet(DATA_MODELS / "09_mk_structural_media_population.parquet")

In [4]:
media_cols = [
    "internet_access",
    "mobile_internet_access",
    "internet_frequency",
    "radio_listener",
    "youtube",
    "facebook",
    "instagram",
    "tiktok",
    "whatsapp",
    "reddit",
    "snapchat",
    "x_twitter",
    "threads",
    "bluesky",
    "truthsocial"
]

In [5]:
# computing national media baseline
national_media_baseline = (
    df[media_cols]
    .multiply(df["pwgtp"], axis=0)
    .sum()
    / df["pwgtp"].sum()
)

national_media_baseline

internet_access           0.946541
mobile_internet_access    0.939734
internet_frequency        1.772246
radio_listener            0.681662
youtube                   0.851719
facebook                  0.729095
instagram                 0.505528
tiktok                    0.401325
whatsapp                  0.331104
reddit                    0.267975
snapchat                  0.291080
x_twitter                 0.206566
threads                   0.091674
bluesky                   0.039250
truthsocial               0.034554
dtype: float64

In [6]:
# reshaping for visualization
baseline_df = (
    national_media_baseline
    .reset_index()
)

baseline_df.columns = ["media_trait", "national_baseline"]

baseline_df

,media_trait,national_baseline
0,internet_access,0.946541
1,mobile_internet_access,0.939734
2,internet_frequency,1.772246
3,radio_listener,0.681662
4,youtube,0.851719
5,facebook,0.729095
6,instagram,0.505528
7,tiktok,0.401325
8,whatsapp,0.331104
9,reddit,0.267975


In [7]:
baseline_path = DATA_PROCESSED / "10_mk_media_national_baseline.csv"

baseline_df.to_csv(baseline_path, index=False)

print("Saved:", baseline_path)

Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/10_mk_media_national_baseline.csv


In [8]:
# computing cluster media means weighted by population
cluster_media_means = (
    df
    .groupby("cluster")
    .apply(
        lambda g: (
            g[media_cols]
            .multiply(g["pwgtp"], axis=0)
            .sum()
            / g["pwgtp"].sum()
        )
    )
)

cluster_media_means

,internet_access,mobile_internet_access,internet_frequency,radio_listener,youtube,facebook,instagram,tiktok,whatsapp,reddit,snapchat,x_twitter,threads,bluesky,truthsocial
cluster,,,,,,,,,,,,,,,
0,0.951088,0.957958,1.717900,0.643506,0.884007,0.738487,0.567181,0.489250,0.410655,0.257945,0.338761,0.214183,0.104173,0.035146,0.033136
1,0.891316,0.868110,2.133342,0.726537,0.720710,0.662549,0.309905,0.279041,0.228425,0.132143,0.164295,0.129767,0.055713,0.026579,0.034872
2,0.980260,0.969448,1.663160,0.717003,0.884133,0.754271,0.529246,0.354193,0.334261,0.299935,0.264875,0.221585,0.087477,0.045833,0.045244
3,0.969424,0.972364,1.628304,0.681479,0.902171,0.775583,0.580408,0.465564,0.401147,0.289856,0.332242,0.218980,0.107145,0.040173,0.031053
4,0.870431,0.852548,2.200173,0.725950,0.686545,0.666337,0.309559,0.259948,0.224413,0.121493,0.147227,0.121615,0.058762,0.025812,0.032969
5,0.965032,0.961288,1.639157,0.646263,0.904632,0.741908,0.572873,0.466502,0.360567,0.343916,0.362343,0.250912,0.115424,0.044685,0.030032
6,0.946566,0.944354,1.696860,0.630777,0.886297,0.714912,0.564888,0.456271,0.325300,0.315872,0.362861,0.231779,0.092184,0.044025,0.033339


In [9]:
# compute deviations from national baseline
cluster_media_deviation = cluster_media_means - national_media_baseline

In [10]:
cluster_media_dev_long = (
    cluster_media_deviation
    .reset_index()
    .melt(
        id_vars="cluster",
        var_name="trait",
        value_name="deviation"
    )
)

cluster_media_dev_long.head()

,cluster,trait,deviation
0,0,internet_access,0.004547
1,1,internet_access,-0.055225
2,2,internet_access,0.033719
3,3,internet_access,0.022884
4,4,internet_access,-0.076110


In [11]:
cluster_media_dev_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   cluster    105 non-null    uint16 
 1   trait      105 non-null    str    
 2   deviation  105 non-null    float64
dtypes: float64(1), str(1), uint16(1)
memory usage: 3.0 KB


In [12]:
cluster_media_dev_long["deviation"].describe()

count    105.000000
mean      -0.004012
std        0.086534
min       -0.195969
25%       -0.047132
50%        0.004775
75%        0.033719
max        0.427927
Name: deviation, dtype: float64

In [13]:
# based on the distribution of deviations, we can set a threshold for "significant" deviation (3%)
threshold = 0.03

cluster_media_signature = (
    cluster_media_dev_long
    .loc[cluster_media_dev_long["deviation"].abs() >= threshold]
    .copy()
)

In [14]:
# sorting
cluster_media_signature["abs_dev"] = cluster_media_signature["deviation"].abs()

cluster_media_signature.sort_values(
    ["cluster","abs_dev"],
    ascending=[True,False],
    inplace=True
)
cluster_media_signature

,cluster,trait,deviation,abs_dev
49,0,tiktok,0.087925,0.087925
56,0,whatsapp,0.079551,0.079551
42,0,instagram,0.061652,0.061652
14,0,internet_frequency,-0.054347,0.054347
70,0,snapchat,0.047681,0.047681
...,...,...,...,...
48,6,instagram,0.059360,0.059360
55,6,tiktok,0.054946,0.054946
27,6,radio_listener,-0.050884,0.050884
69,6,reddit,0.047897,0.047897


### Media Behavior Deviations by Cluster

In this step, media behavior patterns were analyzed to identify platform usage characteristics that distinguish each structural cluster from the national population baseline.

First, a **national media baseline** was computed using the PWGTP population weights from the ACS-enriched synthetic population dataset. This baseline represents the estimated share of the U.S. population using each media platform or exhibiting each media behavior.

Second, **cluster-level weighted means** were calculated for all media variables. Deviations were then computed as:

cluster_mean − national_baseline

This produced a measure of **behavioral over- or under-indexing** for each cluster relative to the national population.

To isolate meaningful signals and remove noise, a **±0.03 deviation threshold (3 percentage points)** was applied, consistent with the filtering logic used previously for psychological traits. Only media behaviors exceeding this threshold were retained as **cluster-defining media signals**.

### Outcome

The resulting table captures the **media behavioral signature of each cluster**, highlighting platforms and behaviors where clusters significantly diverge from the national baseline. These signals reveal distinct patterns of digital engagement across clusters, such as stronger usage of specific social media platforms or lower overall internet activity.

These media signatures will be combined with the previously derived **psychological cluster signatures** to construct the final **audience archetypes** in the next stage of the analysis.

In [15]:
# for better readability, we can combine the trait name with the media usage prefix
cluster_media_signature["trait"] = "media_usage: " + cluster_media_signature["trait"]

In [16]:
# adding direction
cluster_media_signature["direction"] = np.where(
    cluster_media_signature["deviation"] > 0,
    "above baseline",
    "below baseline"
)

In [17]:
# align with psych traits
cluster_media_signature = cluster_media_signature[
    ["cluster","trait","deviation","direction"]
]

In [18]:
cluster_media_signature.head()

,cluster,trait,deviation,direction
49,0,media_usage: tiktok,0.087925,above baseline
56,0,media_usage: whatsapp,0.079551,above baseline
42,0,media_usage: instagram,0.061652,above baseline
14,0,media_usage: internet_frequency,-0.054347,below baseline
70,0,media_usage: snapchat,0.047681,above baseline


In [19]:
psych_path = DATA_MODELS / "08_mk_cluster_psychological_signatures.parquet"

cluster_psych_signature = pd.read_parquet(psych_path)[
    ["cluster", "trait", "deviation", "direction"]
].copy()

print(cluster_psych_signature.shape)
cluster_psych_signature.head()

(31, 4)


,cluster,trait,deviation,direction
0,0,media_engagement: low,0.0751,above baseline
1,0,media_engagement: high,-0.0672,below baseline
2,0,party_alignment: republican,-0.0461,below baseline
3,0,party_alignment: independent,0.0412,above baseline
4,0,ideology: conservative,-0.0343,below baseline


In [20]:
cluster_psych_signature = cluster_psych_signature.copy()
cluster_media_signature = cluster_media_signature.copy()

cluster_psych_signature["source"] = "psych"
cluster_media_signature["source"] = "media"

In [21]:
# merging media and psych signatures
cluster_full_signature = pd.concat(
    [cluster_psych_signature, cluster_media_signature],
    ignore_index=True
)

In [22]:
# sort by signal strength
cluster_full_signature["abs_dev"] = cluster_full_signature["deviation"].abs()

cluster_full_signature.sort_values(
    ["cluster", "abs_dev"],
    ascending=[True, False],
    inplace=True
)

In [23]:
cluster_full_signature.head(20)

,cluster,trait,deviation,direction,source,abs_dev
31,0,media_usage: tiktok,0.087925,above baseline,media,0.087925
32,0,media_usage: whatsapp,0.079551,above baseline,media,0.079551
0,0,media_engagement: low,0.075100,above baseline,psych,0.075100
1,0,media_engagement: high,-0.067200,below baseline,psych,0.067200
33,0,media_usage: instagram,0.061652,above baseline,media,0.061652
34,0,media_usage: internet_frequency,-0.054347,below baseline,media,0.054347
35,0,media_usage: snapchat,0.047681,above baseline,media,0.047681
2,0,party_alignment: republican,-0.046100,below baseline,psych,0.046100
3,0,party_alignment: independent,0.041200,above baseline,psych,0.041200
36,0,media_usage: radio_listener,-0.038155,below baseline,media,0.038155


In [24]:
# ANALYSIS

In [25]:
# selecting top traits per cluster and source
top_psych = (
    cluster_full_signature
    .query("source == 'psych'")
    .groupby("cluster")
    .head(3)
)

top_media = (
    cluster_full_signature
    .query("source == 'media'")
    .groupby("cluster")
    .head(3)
)

In [26]:
top_signals = pd.concat(
    [top_psych, top_media],
    ignore_index=True
)

In [27]:
top_signals.head(20)

,cluster,trait,deviation,direction,source,abs_dev
0,0,media_engagement: low,0.075100,above baseline,psych,0.075100
1,0,media_engagement: high,-0.067200,below baseline,psych,0.067200
2,0,party_alignment: republican,-0.046100,below baseline,psych,0.046100
3,1,media_engagement: low,-0.072500,below baseline,psych,0.072500
4,1,religiosity: none,-0.066300,below baseline,psych,0.066300
5,1,ideology: conservative,0.060500,above baseline,psych,0.060500
6,2,party_alignment: republican,0.052700,above baseline,psych,0.052700
7,2,media_engagement: low,-0.048700,below baseline,psych,0.048700
8,2,media_engagement: high,0.047200,above baseline,psych,0.047200
9,3,media_engagement: low,0.034600,above baseline,psych,0.034600


In [28]:
# sort for readability
top_signals = top_signals.sort_values(
    ["cluster", "abs_dev"],
    ascending=[True, False]
)

In [29]:
# clean summary table
cluster_summary = (
    top_signals
    .groupby("cluster")["trait"]
    .apply(list)
    .reset_index(name="key_traits")
)

cluster_summary

,cluster,key_traits
0,0,"[media_usage: tiktok, media_usage: whatsapp, m..."
1,1,"[media_usage: internet_frequency, media_usage:..."
2,2,"[media_usage: internet_frequency, party_alignm..."
3,3,"[media_usage: internet_frequency, media_usage:..."
4,4,"[media_usage: internet_frequency, media_usage:..."
5,5,"[media_usage: internet_frequency, media_usage:..."
6,6,"[media_usage: internet_frequency, media_usage:..."


### Final Segmentation Table — Full Pipeline Output

This table combines structural population shares, psychological
signatures, and media signatures into a single cluster profile
for each of the 7 adult archetypes. This is the final output
of the societal baseline pipeline and the input to the MK Intel
TA analysis layer.

In [30]:
cluster_pop_path = DATA_MODELS / "06b_mk_structural_population_clustered_v2.parquet"

df_struct = pd.read_parquet(cluster_pop_path)

print("Shape:", df_struct.shape)
df_struct.head()

Shape: (778466, 16)


,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,mar_tier,tenure,household_size,vehicle_count,hhincome_tier,household_type,serialno,sporder,pwgtp,cluster
0,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,No_Rent,2,0,0-19k,housing_unit,2023HU1043211,2,58,5
1,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,No_Rent,4,2,20-49k,housing_unit,2019HU1076190,2,46,3
2,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,Never_Married,Group_Quarters,1,0,group_quarters,group_quarters,2019GQ0046130,1,12,4
3,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Owner,5,2,50-99k,housing_unit,2019HU0403832,1,76,3
4,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,No_Rent,4,1,20-49k,housing_unit,2019HU0277198,1,64,3


In [31]:
# computing weighted cluster population
cluster_population = (
    df_struct
    .groupby("cluster")["pwgtp"]
    .sum()
    .reset_index(name="weighted_population")
)

cluster_population

,cluster,weighted_population
0,0,1728112
1,1,3815257
2,2,5670484
3,3,5874781
4,4,3148523
5,5,6312036
6,6,4335451


In [32]:
# compute population share
cluster_population["population_share"] = (
    cluster_population["weighted_population"]
    / cluster_population["weighted_population"].sum()
)

In [33]:
# keeping final fields
cluster_population = cluster_population[
    ["cluster", "population_share"]
]

cluster_population

,cluster,population_share
0,0,0.055954
1,1,0.123532
2,2,0.183602
3,3,0.190217
4,4,0.101945
5,5,0.204375
6,6,0.140376


In [34]:
# check
cluster_population["population_share"].sum()

np.float64(1.0)

In [35]:
# create psych table
psych_traits = (
    top_signals
    .query("source == 'psych'")
    .groupby("cluster")["trait"]
    .apply(list)
    .reset_index(name="psych_signals")
)

psych_traits

,cluster,psych_signals
0,0,"[media_engagement: low, media_engagement: high..."
1,1,"[media_engagement: low, religiosity: none, ide..."
2,2,"[party_alignment: republican, media_engagement..."
3,3,[media_engagement: low]
4,4,"[religiosity: none, media_engagement: low, rel..."
5,5,"[religiosity: none, media_engagement: low]"
6,6,"[media_engagement: low, media_engagement: high..."


In [36]:
# create media table
media_traits = (
    top_signals
    .query("source == 'media'")
    .groupby("cluster")["trait"]
    .apply(list)
    .reset_index(name="media_signals")
)

media_traits

,cluster,media_signals
0,0,"[media_usage: tiktok, media_usage: whatsapp, m..."
1,1,"[media_usage: internet_frequency, media_usage:..."
2,2,"[media_usage: internet_frequency, media_usage:..."
3,3,"[media_usage: internet_frequency, media_usage:..."
4,4,"[media_usage: internet_frequency, media_usage:..."
5,5,"[media_usage: internet_frequency, media_usage:..."
6,6,"[media_usage: internet_frequency, media_usage:..."


In [37]:
# building final cluster summary table
cluster_profiles = (
    cluster_population
    .merge(psych_traits, on="cluster", how="left")
    .merge(media_traits, on="cluster", how="left")
)

cluster_profiles

,cluster,population_share,psych_signals,media_signals
0,0,0.055954,"[media_engagement: low, media_engagement: high...","[media_usage: tiktok, media_usage: whatsapp, m..."
1,1,0.123532,"[media_engagement: low, religiosity: none, ide...","[media_usage: internet_frequency, media_usage:..."
2,2,0.183602,"[party_alignment: republican, media_engagement...","[media_usage: internet_frequency, media_usage:..."
3,3,0.190217,[media_engagement: low],"[media_usage: internet_frequency, media_usage:..."
4,4,0.101945,"[religiosity: none, media_engagement: low, rel...","[media_usage: internet_frequency, media_usage:..."
5,5,0.204375,"[religiosity: none, media_engagement: low]","[media_usage: internet_frequency, media_usage:..."
6,6,0.140376,"[media_engagement: low, media_engagement: high...","[media_usage: internet_frequency, media_usage:..."


In [38]:
output_path = DATA_PROCESSED / "10_mk_cluster_profiles.parquet"

cluster_profiles.to_parquet(output_path, index=False)

print("Saved:", output_path)

Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/10_mk_cluster_profiles.parquet
